# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def transposee(A):
    """Échange lignes et colonnes."""
    lignes = len(A)
    colonnes = len(A[0])
    resultat = []
    for j in range(colonnes):
        nouvelle_ligne = []
        for i in range(lignes):
            nouvelle_ligne.append(A[i][j])
        resultat.append(nouvelle_ligne)
    return resultat


print("TP4 - Ex 1.1 :", transposee([[1, 2, 3], [4, 5, 6]]))



## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def produit(A, B):
    """Produit de deux matrices."""
    n = len(A)  
    p = len(B) 
    m = len(B[0]) 
    resultat = []
    for i in range(n):
        ligne = []
        for j in range(m):
            somme = 0
            for k in range(p):
                somme = somme + A[i][k] * B[k][j]
            ligne.append(somme)
        resultat.append(ligne)
    return resultat


A = [[1, 2], [3, 4]]
identite = [[1, 0], [0, 1]]
print("TP4 - Ex 2.1 :", produit(A, identite), produit(A, A))
print("A * I == A ->", produit(A, identite) == A)


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def matrice_aleatoire(n):
    return [[random.random() for j in range(n)] for i in range(n)]


print("TP4 - Ex 2.2 :")
temps_precedent = None
for n in (40, 80, 120):
    A = matrice_aleatoire(n)
    B = matrice_aleatoire(n)
    debut = time.perf_counter()
    produit(A, B)
    duree = time.perf_counter() - debut
    print("n =", n, ": temps =", round(duree, 4), "s")
    if temps_precedent is not None and n == 80:
        print("   rapport quand la taille double :", round(duree / temps_precedent, 1))
    temps_precedent = duree


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

n = 120
A = matrice_aleatoire(n)
B = matrice_aleatoire(n)

debut = time.perf_counter()
produit(A, B)
t_main = time.perf_counter() - debut

An = np.array(A)
Bn = np.array(B)
debut = time.perf_counter()
for r in range(100):
    An @ Bn
t_numpy = (time.perf_counter() - debut) / 100

print("TP4 - Ex 3.1 : à la main", round(t_main, 4), "s, numpy", round(t_numpy, 6), "s")
print("numpy est", round(t_main / t_numpy), "fois plus rapide")


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def resoudre(A, b):
    """Résout A x = b par Gauss avec pivot partiel. Renvoie la liste x."""
    n = len(A)
    M = []
    for i in range(n):
        M.append(list(A[i]) + [b[i]])

    for k in range(n):
        ligne_pivot = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[ligne_pivot][k]):
                ligne_pivot = i
        if abs(M[ligne_pivot][k]) < 1e-12:
            raise ValueError("système singulier")
        M[k], M[ligne_pivot] = M[ligne_pivot], M[k]

        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] = M[i][j] - facteur * M[k][j]

    x = [0] * n
    for i in range(n - 1, -1, -1):
        somme = 0
        for j in range(i + 1, n):
            somme = somme + M[i][j] * x[j]
        x[i] = (M[i][n] - somme) / M[i][i]
    return x

A = [[1, 1, 0],
     [1, 0, 1],
     [0, 1, 1]]
b = [2, 3.5, -0.5]
x = resoudre(A, b)
print("TP4 - Ex 4.1 : solution =", x)
print("A x =", [sum(A[i][j] * x[j] for j in range(3)) for i in range(3)])


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

n = 50
A = matrice_aleatoire(n)
b = [random.random() for i in range(n)]
x_moi = resoudre(A, b)
x_numpy = np.linalg.solve(np.array(A), np.array(b))
ecart = max(abs(x_moi[i] - x_numpy[i]) for i in range(n))
print("TP4 - Ex 4.2 : écart maximal =", ecart)


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np


n = 12
H = np.array([[1 / (i + j + 1) for j in range(n)] for i in range(n)])
b = H.sum(axis=1)           
x_solve = np.linalg.solve(H, b)
x_inverse = np.linalg.inv(H) @ b
print("TP4 - Ex 5.1 :")
print("erreur avec solve :", np.max(np.abs(x_solve - 1)))
print("erreur avec inv   :", np.max(np.abs(x_inverse - 1)))
print("conditionnement   :", np.linalg.cond(H))


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def rotation(angle_degres):
    a = math.radians(angle_degres)
    return [[math.cos(a), -math.sin(a)],
            [math.sin(a), math.cos(a)]]


composee = produit(rotation(60), rotation(30))
directe = rotation(90)
print("TP4 - Ex 6.1 :")
print("R(60) * R(30) =", composee)
print("R(90)         =", directe)
ok = all(abs(composee[i][j] - directe[i][j]) < 1e-12 for i in range(2) for j in range(2))
print("Égales à l'arrondi près ->", ok)


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def determinant(A):
    """Déterminant par élimination : produit des pivots, signe changé à chaque
    échange de lignes."""
    n = len(A)
    M = [list(ligne) for ligne in A]
    det = 1
    for k in range(n):
        ligne_pivot = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[ligne_pivot][k]):
                ligne_pivot = i
        if abs(M[ligne_pivot][k]) < 1e-12:
            return 0
        if ligne_pivot != k:
            M[k], M[ligne_pivot] = M[ligne_pivot], M[k]
            det = -det
        det = det * M[k][k]
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] = M[i][j] - facteur * M[k][j]
    return det


A = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]
print("TP4 - Ex 7.1 :", determinant(A), "(numpy :", np.linalg.det(np.array(A)), ")")
print("det [[1,2],[2,4]] =", determinant([[1, 2], [2, 4]]))


## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

n = 12
H = [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]
b = [sum(ligne) for ligne in H] 

try:
    resoudre(H, b)
except ValueError as erreur:
    print("TP4 - Ex 8.1 : premier essai ->", erreur)

def resoudre_exact(A, b):
    """Comme resoudre, mais le test de singularité est 'pivot == 0'."""
    n = len(A)
    M = [list(A[i]) + [b[i]] for i in range(n)]
    for k in range(n):
        ligne_pivot = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[ligne_pivot][k]):
                ligne_pivot = i
        if M[ligne_pivot][k] == 0:
            raise ValueError("système singulier")
        M[k], M[ligne_pivot] = M[ligne_pivot], M[k]
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] = M[i][j] - facteur * M[k][j]
    x = [0] * n
    for i in range(n - 1, -1, -1):
        somme = 0
        for j in range(i + 1, n):
            somme = somme + M[i][j] * x[j]
        x[i] = (M[i][n] - somme) / M[i][i]
    return x


x = resoudre_exact(H, b)
print("Solution exacte :", [str(v) for v in x])

## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [ ]:
import math
import random
import time
from fractions import Fraction

import numpy as np

def translation(tx, ty):
    return [[1, 0, tx],
            [0, 1, ty],
            [0, 0, 1]]


def rotation_h(angle_degres):
    a = math.radians(angle_degres)
    return [[math.cos(a), -math.sin(a), 0],
            [math.sin(a), math.cos(a), 0],
            [0, 0, 1]]


def appliquer(M, point):
    """Applique la matrice 3x3 M au point (x, y) (coordonnées homogènes)."""
    colonne = [[point[0]], [point[1]], [1]]
    r = produit(M, colonne)
    return (round(r[0][0], 6), round(r[1][0], 6))


T = translation(3, 1)
R = rotation_h(90)
translation_puis_rotation = produit(R, T)
rotation_puis_translation = produit(T, R)
print("TP4 - Ex 9.1 :")
print("translation puis rotation :", appliquer(translation_puis_rotation, (1, 0)))
print("rotation puis translation :", appliquer(rotation_puis_translation, (1, 0)))

